Dans cet atelier nous allons entrainer un modèle machine learning avec Scikit-Learn et puis le déployer en tant que web service local et aussi en tant que web service ACI "Azure Container Instance". 

Les étapes de cet Atelier sont les suivantes: 

1. entrainer localement un modèle Scikit-learn
2. Suivre les expérimentations Scikit-learn avec MLFlow sur Azure Machine Learning 
3. Enregistrer le modèle sur Azure Machine Learning 
4. Déployer et tester le modèle en tant que web service local 
5. Déployer et tester le modèle en tant que web service ACI "Azure Container Instance"

Tout au long de cet atelier nous allons utiliser Pima Indians Diabetes Database. Pour de plus amples informations sur les colonnes, visitez le lien ci-dessous: 

https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

Smith, J.W., Everhart, J.E., Dickson, W.C., Knowler, W.C., & Johannes, R.S. (1988). Using the ADAP learning algorithm to forecast the onset of diabetes mellitus. In Proceedings of the Symposium on Computer Applications and Medical Care (pp. 261--265). IEEE Computer Society Press.

## 1. L'entrainement du modèle Scikit-Learn 

Assurez vous que vous avez installé anaconda avec une version de python 3.7.*

Assurez vous aussi que vous avez Docker installé : https://docs.docker.com/desktop/install/windows-install/ 

Dans le dossier de l'atelier, installez les dépendances requises: pip install -r requirements.txt 

* azureml-core==1.39
* pandas==1.3.5
* scikit-learn==0.24.2
* cloudpickle==2.0.0
* psutil==5.9.0
* mlflow==1.24.0

In [32]:
import mlflow
from azureml.core import Workspace
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer


In [33]:
print(mlflow.__version__)


2.22.4


In [34]:
import warnings
warnings.filterwarnings('ignore')


Pour se connecter au workspace Azure machine learning, deux cas se présentent: 

1. Télecharger le fichier de configuration et appeler le en utilisant ws = Workspace.from_config() 

![image.png](attachments/image2.png)

2. Ou bien en entrant manuellement les informations nécessaires, comme indiqué dans la cellule suivante.

In [35]:
ws = Workspace.from_config() 
# subscription_id = "25ba4898-4765-4550-90a3-d21b511dcaac"
# resource_group = 'Lab3'
# workspace_name = 'lab3'
# workspace_location="France Central"
# ws = Workspace.create(name = workspace_name,location = workspace_location,resource_group = resource_group, subscription_id = subscription_id,exist_ok=True)


#### Mettre en place un URI de Tracking des experimentation MLflow sur Azure

Nous indiquons à mlflow que l'URI de tracking est celui de mlflow dans azure ML. Toutefois, le lien de tracking de Mlflow dans azure à la forme suivante:

azureml://<region>.api.azureml.ms/mlflow/v1.0/subscriptions/<subscription-id>/resourceGroups/<resource-group>/providers/Microsoft.MachineLearningServices/workspaces/<aml-workspace>?

In [36]:
mlflow.set_tracking_uri(ws.get_mlflow_tracking_uri())


Nous allons créer une expérimentation Mlflow dont le nom est "diabetes-sklearn" 

In [6]:
experiment_name = 'diabetes-sklearn'
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='', creation_time=1765835137352, experiment_id='2de525ef-9a21-43ac-ac04-3e2e644e49ac', last_update_time=None, lifecycle_stage='active', name='diabetes-sklearn', tags={}>

#### Chargement du dataset et traitements necéssaires 

In [7]:
df=pd.read_csv('diabetes.csv')


In [9]:
df.shape


(768, 9)

In [9]:
df.columns


Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [10]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [11]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
df_scaled=scaler.fit_transform(df)


In [12]:
df_scaled


array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.46849198,
         1.4259954 ,  1.36589591],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.36506078,
        -0.19067191, -0.73212021],
       [ 1.23388019,  1.94372388, -0.26394125, ...,  0.60439732,
        -0.10558415,  1.36589591],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.68519336,
        -0.27575966, -0.73212021],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.37110101,
         1.17073215,  1.36589591],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.47378505,
        -0.87137393, -0.73212021]], shape=(768, 9))

In [13]:
x=df.iloc[:,:-1]
y=df.iloc[:,-1:]


In [14]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)


In [15]:
print("X_train:",x_train.shape)
print("X_test:",x_test.shape)
print("Y_train:",y_train.shape)
print("Y_test:",y_test.shape)


X_train: (614, 8)
X_test: (154, 8)
Y_train: (614, 1)
Y_test: (154, 1)


In [16]:
rf_clf = RandomForestClassifier() 


Au lieu d'appeler a chaque fois une fonction de logging (log_metric, log_artifact, ...) nous allons utiliser la fonction autolog() afin de logger toutes les informations relatives au modèle (metrics, parameters, ...) 

les frameworks suivants supporte la fonction autolog():

    Scikit-learn

    TensorFlow and Keras

    Gluon

    XGBoost

    LightGBM

    Statsmodels

    Spark

    Fastai

    Pytorch



In [18]:
mlflow.sklearn.autolog(max_tuning_runs=None)


2025/12/15 23:10:40 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.7.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.


#### Hyperparameter tuning 

Nous alons utiliser l'algorithme GridSearchCV pour trouver les meilleurs paramétres

In [19]:
param_grid = {'n_estimators': [10,20,30], 'max_depth':[2,7,10]}


In [20]:
gridcv=GridSearchCV(rf_clf,param_grid=param_grid,scoring = ['roc_auc', 'precision', 'recall', 'f1', 'accuracy'],refit = 'roc_auc',cv=10)


In [ ]:
gridcv.fit(x_train, y_train)


2025/12/15 23:10:45 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'ad7473e0-b80e-4adf-ad2d-1838d60104db', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run silver_oxygen_cxcn6glh at: https://francecentral.api.azureml.ms/mlflow/v2.0/subscriptions/995da238-d678-4991-b65a-b3cb2d444d81/resourceGroups/MLOps-labs/providers/Microsoft.MachineLearningServices/workspaces/MLOps-labs/#/experiments/2de525ef-9a21-43ac-ac04-3e2e644e49ac/runs/dadd87f4-0e94-445f-8edb-c8db42d4984b
🧪 View experiment at: https://francecentral.api.azureml.ms/mlflow/v2.0/subscriptions/995da238-d678-4991-b65a-b3cb2d444d81/resourceGroups/MLOps-labs/providers/Microsoft.MachineLearningServices/workspaces/MLOps-labs/#/experiments/2de525ef-9a21-43ac-ac04-3e2e644e49ac
🏃 View run salmon_knot_8tmr9jlj at: https://francecentral.api.azureml.ms/mlflow/v2.0/subscriptions/995da238-d678-4991-b65a-b3cb2d444d81/resourceGroups/MLOps-labs/providers/Microsoft.MachineLearningServices/workspaces/MLOps-labs/#/experiments/2de525ef-9a21-43ac-ac04-3e2e644e49ac/runs/250cb6b1-17e0-48e6-bf86-197ca145ba73
🧪 View experiment at: https://francecentral.api.azureml.ms/mlflow/v2.0/subscriptions/995da2

,estimator,RandomForestClassifier()
,param_grid,"{'max_depth': [2, 7, ...], 'n_estimators': [10, 20, ...]}"
,scoring,"['roc_auc', 'precision', ...]"
,n_jobs,None
,refit,'roc_auc'
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,30


#### Visualisation des résultats sur Azure ML experiments

Les résultats du tracking et logging avec Mlflow peuvent être visualisés sur Azure machine learning studio en cliquant sur "Jobs" et puis sur le nom de l'experimentation (pour notre cas: diabetes-sklearn)

![](attachments/image3.png)

En cliquant sur le dernier job vous pouvez voir les résultats Mlflow de notre logging 

![](attachments/image4.png)

les artifacts et run_id du meilleur modèle peuvent être trouvés en cliquent sur l'onglet "output+logs"

![](attachments/image5.png)


In [ ]:
# modelLogistic = LogisticRegression()
# modelLogistic.fit(x_train,y_train)


In [31]:
# parameters={'penalty':['l1','l2'],"C":[1,2,3,4,5,10,100],'max_iter':[10,20,30]}


In [32]:
# mlflow.sklearn.autolog(max_tuning_runs=None)
# gridcv=GridSearchCV(modelLogistic,param_grid=parameters,scoring = ['roc_auc', 'precision', 'recall', 'f1', 'accuracy'],refit = 'roc_auc',cv=10)


In [33]:
# gridcv.fit(x_train,y_train)

## 2.  L'enregistrement du modèle 
L'enregistrement du modèle dans le registre de modèles d'Azure Machine Learning a pour but de permettre aux utilisateurs de suivre les modifications apportées au modèle via la gestion des versions du modèle.

Nous allons trouver l'expérimentation à partir du workspace en définant le workspace et le nom de l'expérimentation.


In [34]:
from azureml.core import Experiment, Workspace
experiment_name = 'diabetes-sklearn'
experiment = Experiment(ws, experiment_name)


In [36]:
# Afficher tous les runs 
for r in experiment.get_runs():
    print(r)
    

Run(Experiment: diabetes-sklearn,
Id: ad7473e0-b80e-4adf-ad2d-1838d60104db,
Type: None,
Status: Completed)


Trouver le Run actuelle 

In [38]:
run_id = 'ad7473e0-b80e-4adf-ad2d-1838d60104db'
run = [r for r in experiment.get_runs() if r.id == run_id][0]


Enregistrer le modèle 

    model_name: un nom arbitraire pour enregister le modèle 
    model_path: chemin vers  model.pkl

In [39]:
model = run.register_model(model_name = 'diabetes_model', model_path = 'best_estimator/model.pkl')


On peut visualiser nos modèle enregistés en visitant "Models"

![image.png](attachments/image6.png)


## 3. la création d'un script de Scoring

Le script de scoring généralement appelé score.py est utilisé lors de l'inférence comme point d'entrée du modèle.

score.py (voir le dossier de l'atelier) consiste en deux fonctions obligatoires:

1. init(): charge le modèle en tant que variable globale
2. run(): reçoit les nouvelles données à prédire à travers le paramètre data
      * effectue un pré-traitement des nouvelles données (optionnel)
      * effectue une prédiction sur les nouvelles données
      * effectue un post-traitement sur les prédictions (optionnel)
      * renvoie les résultats de la prédiction

## 4. Déploiement en local

Maintenant nous allons debuger le web service locallement avant de le déployer en production avec ACI " Azure container Instance"

Récupérez le modèle enregistré en définissant le workspace, le nom du modèle et la version du modèle.

In [13]:
from azureml.core.model import Model
model = Model(ws, 'diabetes_model', version=1)


### la création d'un environnement d'inférence personnalisé 
Lors de l'entrainement du modèle, nous avons enregistré les dépendances de l'environnement dans MLFlow sous la forme d'un fichier conda.yaml. Nous utiliserons ce fichier pour créer un environnement d'inférence personnalisé.

![image.png](attachments/image7.png)

Téléchargez le fichier conda.yml vers le dossier de l'atelier (déjà faite pour vous) et rajoutant d'autres dépendances comme: azureml-defaults, applicationinsights.

Nous allons nous baser sur ce fichier pour créer un environnement personnalisé.

Après enregistrement de cet environnement, nous pouvons le consulter dans la section "Environment" dans Azure Machine Learning et plus particulièrement dans l'onglet "Custom environments".

![image-2.png](attachments/image8.png)


In [41]:
from azureml.core import Environment
env = Environment.from_conda_specification(name='diabetes-env', file_path="./conda.yaml")
env.python.conda_dependencies.set_python_version("3.9")
env.python.conda_dependencies.add_pip_package("azureml-defaults")
env.python.conda_dependencies.add_pip_package("azureml-inference-server-http")
env.register(ws)


{
    "assetId": "azureml://locations/francecentral/workspaces/5d4d3c8a-74a5-44e8-bcdb-020f62bb4931/environments/diabetes-env/versions/5",
    "databricks": {
        "eggLibraries": [],
        "jarLibraries": [],
        "mavenLibraries": [],
        "pypiLibraries": [],
        "rcranLibraries": []
    },
    "docker": {
        "arguments": [],
        "baseDockerfile": null,
        "baseImage": "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:20251130.v1",
        "baseImageRegistry": {
            "address": null,
            "password": null,
            "registryIdentity": null,
            "username": null
        },
        "buildContext": null,
        "enabled": false,
        "platform": {
            "architecture": "amd64",
            "os": "Linux"
        },
        "sharedVolumes": true,
        "shmSize": null
    },
    "environmentVariables": {
        "EXAMPLE_ENV_VAR": "EXAMPLE_VALUE"
    },
    "inferencingStackVersion": null,
    "name": "diabetes-env",
   

### Définons la configuration d'inférence 

In [42]:
from azureml.core.model import InferenceConfig
inference_config = InferenceConfig(
    environment=env,
    source_directory=".",
    entry_script="./score.py",
)


### Définons la configuration de déploiement

In [16]:
from azureml.core.webservice import LocalWebservice
deployment_config = LocalWebservice.deploy_configuration(port=6789)


### Déployons le service localement

Avant d'exécuter la cellule ci-dessous, assurez vous que Docker est démarré 

In [17]:
service = Model.deploy(
    workspace = ws,
    name = 'diabetes-prediction-service',
    models = [model],
    inference_config = inference_config,
    deployment_config = deployment_config,
    overwrite=True)


Warning, azureml-defaults not detected in provided environment pip dependencies. The azureml-defaults package contains requirements for the inference stack to run, and should be included.


Generating Docker build context.
======Starting Image Build on Compute======
The run ID for the image build on compute is imgbldrun_6ac1ea0
Additional logs for the run: https://ml.azure.com/experiments/id/prepare_image/runs/imgbldrun_6ac1ea0?wsid=/subscriptions/995da238-d678-4991-b65a-b3cb2d444d81/resourcegroups/MLOps-labs/workspaces/MLOps-labs&tid=39626157-a047-4689-87a2-6fa645cb5cb7
2025-12-15T23:47:21: Logging into Docker registry: 5d4d3c8a74a544e8bcdb020f62bb4931.azurecr.io
2025-12-15T23:47:21: WARNING! Using --password via the CLI is insecure. Use --password-stdin.

2025-12-15T23:47:21: Login Succeeded
2025-12-15T23:47:21: WARNING! Your credentials are stored unencrypted in '/root/.docker/config.json'.
2025-12-15T23:47:21: Configure a credential helper to remove this warning. See
2025-12-15T23:47:21: https://docs.docker.com/go/credential-store/



2025-12-15T23:47:22: Running: ['docker', 'build', '-f', 'azureml-environment-setup/Dockerfile', '.', '-t', '5d4d3c8a74a544e8bcdb020f62b

In [18]:
service.wait_for_deployment(show_output=True)


Checking container health...
Local webservice is running at http://localhost:6789


Le fichier model.pkl sera téléchargé à partir d'Azure Machine Learning dans un dossier local temporaire et une image Docker avec les dépendances est créée et enregistrée dans Azure Container Registry (ACR). L'image sera téléchargée d'ACR sur la machine locale et un conteneur Docker exécutant le service Web est construit à partir de l'image localement.



Pour avoir l'URI de scoring lancez la commande suivante 

In [ ]:
print(service.scoring_uri)


http://localhost:6789


## 5. Testons le service localement 

Pour tester le service localement, ouvrez le notebook "Inference_test.ipynb" et assigner l'URI de scroring à la variable scoring_uri

Nous avons envoyé une demande d'inference au scoring_uri avec les données au format JSON. Voici à quoi ressemble input_data :

{"input": "[{\"Pregnancies\":6,\"Glucose\":148,\"BloodPressure\":72,\"SkinThickness\":35,\"Insulin\":0,\"BMI\":33.6,\"DiabetesPedigreeFunction\":0.627,\"Age\":50}]"}


Voici un exemple de la réponse pour l'inférence sur un seul enregistrement. La valeur de retour contient la probabilité qu'une personne reçoive un diagnostic de diabète.

prediction: "{\"proba\": [0.4829951216899814]}"


Le format de réponse peut être personnalisé dans la fonction run du fichier score.py.

## 6. Déplyons le service sur ACI

In [56]:
from azureml.core import Workspace
ws = Workspace.from_config()
from azureml.core.model import Model
model = Model(ws, 'diabetes_model', version=1)


In [58]:
from azureml.core import Environment
from azureml.core.conda_dependencies import CondaDependencies
conda_dep = CondaDependencies()
conda_dep.set_python_version("3.10")
conda_dep.add_pip_package("azureml-defaults")
conda_dep.add_pip_package("azureml-inference-server-http")
conda_dep.add_pip_package("numpy<2.0")
conda_dep.add_pip_package("pandas")
conda_dep.add_pip_package("scikit-learn")
conda_dep.add_pip_package("mlflow")
env = Environment(name='diabetes-prod-clean')
env.python.conda_dependencies = conda_dep
env.register(ws)


{
    "assetId": "azureml://locations/francecentral/workspaces/5d4d3c8a-74a5-44e8-bcdb-020f62bb4931/environments/diabetes-prod-clean/versions/1",
    "databricks": {
        "eggLibraries": [],
        "jarLibraries": [],
        "mavenLibraries": [],
        "pypiLibraries": [],
        "rcranLibraries": []
    },
    "docker": {
        "arguments": [],
        "baseDockerfile": null,
        "baseImage": "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:20251130.v1",
        "baseImageRegistry": {
            "address": null,
            "password": null,
            "registryIdentity": null,
            "username": null
        },
        "buildContext": null,
        "enabled": false,
        "platform": {
            "architecture": "amd64",
            "os": "Linux"
        },
        "sharedVolumes": true,
        "shmSize": null
    },
    "environmentVariables": {
        "EXAMPLE_ENV_VAR": "EXAMPLE_VALUE"
    },
    "inferencingStackVersion": null,
    "name": "diabetes-pr

In [59]:
from azureml.core.model import InferenceConfig
inference_config = InferenceConfig(
    environment=env,
    source_directory=".",
    entry_script="./score.py")


In [60]:
from azureml.core.webservice import AciWebservice
deployment_config = AciWebservice.deploy_configuration(cpu_cores=0.1, memory_gb=0.5, auth_enabled=False)


In [61]:
service = Model.deploy(
    workspace=ws,
    name='diabetes-prediction-service5',
    models=[model],
    inference_config=inference_config,
    deployment_config=deployment_config,
    overwrite=True
)

service.wait_for_deployment(show_output=True)


Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2025-12-17 22:34:48+01:00 Creating Container Registry if not exists.
2025-12-17 22:34:48+01:00 Registering the environment.
2025-12-17 22:34:49+01:00 Building image..
2025-12-17 22:47:19+01:00 Generating deployment configuration.
2025-12-17 22:47:21+01:00 Submitting deployment to compute..
2025-12-17 22:47:25+01:00 Checking the status of deployment diabetes-prediction-service5..
2025-12-17 22:50:09+01:00 Checking the status of inference endpoint diabetes-prediction-service5.
Succeeded
ACI service creation operation finished, operation "Succeeded"


In [ ]:
# service.delete()


In [63]:
print(service.scoring_uri)


http://c57f3433-708b-42e9-ae0c-7da448779551.francecentral.azurecontainer.io/score


### Testons le service 

Refaites la même chose en ouvrant le notebook "Inference_test.ipynb" et assigner le nouveau URI de scroring à la variable scoring_uri 

## Déploiement sur AKS "Azure Kubernetes Service" 

ACI est recommandé pour les tests sur une petite charge de travail de production. Pour une charge de travail importante azure fourni AKS "Azure Kubernetes Service"

Malheureusement le Déploiement sur AKS n'est pas pris en compte  avec notre sousription actuelle. 

Néanmoins, je vous ai fourni le code de déploiement pour une éventuelle utilisation future.

In [ ]:
from azureml.core.webservice import AksWebservice
deployment_config = AksWebservice.deploy_configuration(cpu_cores=1, memory_gb=1, auth_enabled=False)


In [ ]:
# Ca suppose la création d'un kubernetes cluster 
from azureml.core.compute import AksCompute
aks_target = AksCompute(ws,"myaks")


In [ ]:
service = Model.deploy(
    workspace = ws,
    name = 'diabetes-prediction-service-aks',
    models = [model],
    inference_config = inference_config,
    deployment_config = deployment_config,
    deployment_target=aks_target,
    overwrite=True)
    
service.wait_for_deployment(show_output=True)
